In [1]:
!pip install segmentation-models-pytorch albumentations

import os
import numpy as np
import cv2
import torch
import gdown
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, jaccard_score
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 15.3 MB/s eta 0:00:00
Using device: cuda


In [2]:
# 1. Download
file_id = '1Bv7zmjJVvE7uQbbjwRa-3qn-BI8h99KI'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'data.zip'

if not os.path.exists(output):
    gdown.download(url, output, quiet=False)

# 2. Unzip
if os.path.exists(output) and not os.path.exists("/content/data/data"):
    print("Unzipping...")
    !unzip -o -q data.zip -d /content/data
    print("Data ready at /content/data/data/")
else:
    print("Data already extracted.")

# Set Paths
IMG_PATH = "/content/data/data/images"
MASK_PATH = "/content/data/data/masks"

Downloading...
From: https://drive.google.com/uc?id=1Bv7zmjJVvE7uQbbjwRa-3qn-BI8h99KI
To: /content/data.zip
100%|██████████| 18.0M/18.0M [00:00<00:00, 252MB/s]


Unzipping...
Data ready at /content/data/data/


In [3]:
# --- 1. Augmentation Pipelines ---
train_transform = A.Compose([
    A.Resize(height=256, width=256, interpolation=cv2.INTER_LINEAR),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=15, p=0.3, border_mode=cv2.BORDER_CONSTANT),
    A.ToFloat(max_value=255.0),
    ToTensorV2(),
], additional_targets={'mask': 'mask'})

val_transform = A.Compose([
    A.Resize(height=256, width=256, interpolation=cv2.INTER_LINEAR),
    A.ToFloat(max_value=255.0),
    ToTensorV2(),
], additional_targets={'mask': 'mask'})

# --- 2. Dataset Class ---
class CustomDataset(Dataset):
    def __init__(self, image_dir, mask_dir, file_list, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = file_list
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)
        
        image = np.array(Image.open(img_path).convert("RGB"))
        mask = np.array(Image.open(mask_path).convert("L"))
        
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
            # Add channel dim and threshold
            mask = mask.unsqueeze(0).float()
            mask = (mask > 0.5).float()

        return image, mask

In [4]:
def run_experiment(seed, epochs=20):
    print(f"\n{'='*40}")
    print(f"STARTING RUN WITH SEED: {seed}")
    print(f"{'='*40}")
    
    # 1. Split Data (Random State = Seed)
    all_files = sorted(os.listdir(IMG_PATH))
    train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=seed)
    
    # 2. Loaders
    train_dataset = CustomDataset(IMG_PATH, MASK_PATH, train_files, transform=train_transform)
    test_dataset = CustomDataset(IMG_PATH, MASK_PATH, test_files, transform=val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    # 3. Initialize Pure UNet
    # Changed from UnetPlusPlus to Unet
    model = smp.UnetPlusPlus(
        encoder_name="resnet34", 
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
    ).to(device)
    
    loss_fn = smp.losses.DiceLoss(smp.losses.BINARY_MODE, from_logits=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    
    # 4. Training Loop
    print(f"Training for {epochs} epochs...")
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            
            logits = model(images)
            loss = loss_fn(logits, masks)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
        # Optional: Print every 5 epochs to keep logs clean
        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(train_loader):.4f}")
            
    return model, test_loader

In [5]:
def evaluate_model(model, test_loader):
    model.eval()
    scores = {"dice": [], "iou": [], "prec": [], "rec": [], "acc": []}
    
    with torch.no_grad():
        for image, true_mask in test_loader:
            input_batch = image.to(device)
            logits = model(input_batch)
            probs = logits.sigmoid()
            pred_mask = (probs > 0.5).float()
            
            # Convert to Numpy
            p = pred_mask.cpu().squeeze().numpy().flatten().astype(np.uint8)
            g = true_mask.cpu().squeeze().numpy().flatten().astype(np.uint8)
            
            # Calculate Metrics
            scores["dice"].append(f1_score(g, p, zero_division=1))
            scores["iou"].append(jaccard_score(g, p, zero_division=1))
            scores["prec"].append(precision_score(g, p, zero_division=1))
            scores["rec"].append(recall_score(g, p, zero_division=1))
            scores["acc"].append(accuracy_score(g, p))
            
    # Return average of this run
    return {k: np.mean(v) for k, v in scores.items()}

In [6]:
# --- Configuration ---
SEEDS = [42, 0, 123, 7, 2026] # 5 Runs
EPOCHS_PER_RUN = 50           # Adjust based on your time constraints

# Storage for results
all_results = {"dice": [], "iou": [], "prec": [], "rec": [], "acc": []}

for seed in SEEDS:
    # 1. Train
    model, test_loader = run_experiment(seed, epochs=EPOCHS_PER_RUN)
    
    # 2. Evaluate
    metrics = evaluate_model(model, test_loader)
    
    # 3. Store
    for k, v in metrics.items():
        all_results[k].append(v)
        
    print(f"Run {seed} Results -> Dice: {metrics['dice']:.4f} | IoU: {metrics['iou']:.4f}")

# --- Final Report ---
print("\n" + "="*40)
print("FINAL MULTI-RUN REPORT (Pure UNet)")
print("="*40)
print(f"{'Metric':<10} | {'Mean':<8} | {'Std Dev':<8}")
print("-" * 30)

for k, v in all_results.items():
    mean_val = np.mean(v)
    std_val = np.std(v)
    print(f"{k.upper():<10} | {mean_val:.4f}   | {std_val:.4f}")


STARTING RUN WITH SEED: 42


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Training for 50 epochs...
Epoch 5/50 | Loss: 0.5332
Epoch 10/50 | Loss: 0.3596
Epoch 15/50 | Loss: 0.2331
Epoch 20/50 | Loss: 0.1843
Epoch 25/50 | Loss: 0.1966
Epoch 30/50 | Loss: 0.1544
Epoch 35/50 | Loss: 0.1227
Epoch 40/50 | Loss: 0.1147
Epoch 45/50 | Loss: 0.1437
Epoch 50/50 | Loss: 0.1133
Run 42 Results -> Dice: 0.6903 | IoU: 0.6097

STARTING RUN WITH SEED: 0
Training for 50 epochs...
Epoch 5/50 | Loss: 0.5426
Epoch 10/50 | Loss: 0.3668
Epoch 15/50 | Loss: 0.2240
Epoch 20/50 | Loss: 0.2105
Epoch 25/50 | Loss: 0.1301
Epoch 30/50 | Loss: 0.1239
Epoch 35/50 | Loss: 0.1136
Epoch 40/50 | Loss: 0.1122
Epoch 45/50 | Loss: 0.0938
Epoch 50/50 | Loss: 0.0829
Run 0 Results -> Dice: 0.7118 | IoU: 0.6300

STARTING RUN WITH SEED: 123
Training for 50 epochs...
Epoch 5/50 | Loss: 0.5763
Epoch 10/50 | Loss: 0.3270
Epoch 15/50 | Loss: 0.2098
Epoch 20/50 | Loss: 0.1662
Epoch 25/50 | Loss: 0.1448
Epoch 30/50 | Loss: 0.1575
Epoch 35/50 | Loss: 0.1328
Epoch 40/50 | Loss: 0.0988
Epoch 45/50 | Loss: 0.09

In [7]:
import time
import torch
import numpy as np

# --- Configuration ---
# Uses the 'model' and 'device' variables from your previous cells
ITERATIONS = 100  # Number of times to run for average
WARMUP = 50       # Number of dummy runs to wake up GPU

print(f"⏱️ Benchmarking Inference Speed on {device}...")
print(f"   Model: {model.__class__.__name__}")
print(f"   Input: (1, 3, 256, 256)")

# 1. Prepare Dummy Input
dummy_input = torch.randn(1, 3, 256, 256).to(device)
model.eval()

# 2. Warmup Phase (Crucial for GPU stability)
print(f"   Warming up for {WARMUP} steps...")
with torch.no_grad():
    for _ in range(WARMUP):
        _ = model(dummy_input)

# 3. Measurement Phase
timings = []
print(f"   Measuring over {ITERATIONS} runs...")

with torch.no_grad():
    for _ in range(ITERATIONS):
        # Synchronize GPU to get accurate start/end times
        if device.type == 'cuda':
            torch.cuda.synchronize()
            start = time.time()
            _ = model(dummy_input)
            torch.cuda.synchronize()
            end = time.time()
        else:
            start = time.time()
            _ = model(dummy_input)
            end = time.time()
        
        timings.append((end - start) * 1000) # Convert to milliseconds

# 4. Report
avg_ms = np.mean(timings)
std_ms = np.std(timings)
fps = 1000 / avg_ms

print("\n" + "="*30)
print("🚀 FINAL SPEED RESULTS")
print("="*30)
print(f"Avg Time: {avg_ms:.2f} ms ± {std_ms:.2f} ms")
print(f"FPS:      {fps:.2f} frames/sec")
print("="*30)

⏱️ Benchmarking Inference Speed on cuda...
   Model: UnetPlusPlus
   Input: (1, 3, 256, 256)
   Warming up for 50 steps...
   Measuring over 100 runs...

🚀 FINAL SPEED RESULTS
Avg Time: 8.30 ms ± 0.42 ms
FPS:      120.47 frames/sec
